In [1]:
# -*- coding: utf-8 -*-
"""
RNN Encoder-Decoder Translation Demo
Dataset: English -> Spanish (from TensorFlow Datasets, small subset)
"""

'\nRNN Encoder-Decoder Translation Demo\nDataset: English -> Spanish (from TensorFlow Datasets, small subset)\n'

### Problem Statement

The goal of this project is to develop a sequence-to-sequence (Seq2Seq) model using an RNN Encoder–Decoder architecture for neural machine translation. Specifically, we will build a model that can translate sentences from Portuguese to English using the TED Talks Translation dataset available in TensorFlow Datasets.

#### Objectives:

Preprocess and tokenize bilingual sentence pairs (Portuguese → English).

Train an Encoder–Decoder network where:

- The encoder reads the source (Portuguese) sentence and compresses it into a context vector.

- The decoder generates the target (English) sentence word by word, using teacher forcing during training.

- Evaluate the model’s ability to translate unseen Portuguese sentences into meaningful English outputs.



In [2]:
#import subprocess
#subprocess.run(["pip", "install", "tensorflow_datasets"], capture_output=True)

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from tensorflow import keras

# ---- Load Dataset ----
try:
    dataset, metadata = tfds.load(
        "ted_hrlr_translate/pt_to_en",
        with_info=True,
        as_supervised=True
    )
    train_examples, val_examples = dataset["train"], dataset["validation"]
except Exception as e:
    print(f"TFDS download failed: {e}")
    print("Using a small built-in Portuguese-English dataset instead.")

    fallback_pairs = [
        ("Olá", "hello"),
        ("bom dia", "good morning"),
        ("boa tarde", "good afternoon"),
        ("boa noite", "good night"),
        ("como você está?", "how are you?"),
        ("eu estou bem", "i am fine"),
        ("qual é o seu nome?", "what is your name?"),
        ("meu nome é ana", "my name is ana"),
        ("onde fica a estação?", "where is the station?"),
        ("eu gosto de café", "i like coffee"),
        ("você fala inglês?", "do you speak english?"),
        ("eu não entendo", "i do not understand"),
        ("quanto custa isso?", "how much does this cost?"),
        ("obrigado", "thank you"),
        ("de nada", "you are welcome"),
        ("até logo", "see you later"),
        ("eu moro no brasil", "i live in brazil"),
        ("hoje está chovendo", "it is raining today"),
        ("o livro está na mesa", "the book is on the table"),
        ("ela gosta de música", "she likes music"),
        ("nós vamos para a escola", "we are going to school"),
        ("eles estão felizes", "they are happy"),
        ("eu preciso de ajuda", "i need help"),
        ("olá, como você está?", "hello, how are you?")
    ]

    pt_texts = [pt for pt, _ in fallback_pairs]
    en_texts = [en for _, en in fallback_pairs]

    train_examples = tf.data.Dataset.from_tensor_slices((pt_texts, en_texts))
    val_examples = tf.data.Dataset.from_tensor_slices((pt_texts[:6], en_texts[:6]))
    metadata = None

# ---- Tokenizers ----
def _to_text(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)

tokenizer_en = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    (_to_text(en.numpy()) for _, en in train_examples),
    target_vocab_size=2**13
)

tokenizer_pt = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    (_to_text(pt.numpy()) for pt, _ in train_examples),
    target_vocab_size=2**13
)

vocab_size_en = tokenizer_en.vocab_size + 2
vocab_size_pt = tokenizer_pt.vocab_size + 2
max_len = 40

# ---- Encode Function ----
def encode(pt, en):
    pt = _to_text(pt.numpy())
    en = _to_text(en.numpy())

    pt_tokens = [tokenizer_pt.vocab_size] + tokenizer_pt.encode(pt) + [tokenizer_pt.vocab_size + 1]
    en_tokens = [tokenizer_en.vocab_size] + tokenizer_en.encode(en) + [tokenizer_en.vocab_size + 1]
    return pt_tokens, en_tokens

def tf_encode(pt, en):
    pt, en = tf.py_function(encode, [pt, en], [tf.int64, tf.int64])
    pt.set_shape([None])
    en.set_shape([None])
    return pt, en

# ---- Prepare Data ----
BUFFER_SIZE = 20000
BATCH_SIZE = 64

train_dataset = (
    train_examples.map(tf_encode)
    .filter(lambda x, y: tf.logical_and(tf.size(x) <= max_len, tf.size(y) <= max_len))
    .cache()
    .shuffle(BUFFER_SIZE)
    .padded_batch(BATCH_SIZE, padded_shapes=([None], [None]))
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_examples.map(tf_encode)
    .filter(lambda x, y: tf.logical_and(tf.size(x) <= max_len, tf.size(y) <= max_len))
    .padded_batch(BATCH_SIZE, padded_shapes=([None], [None]))
)

# ---- Build Seq2Seq Model ----
embed_dim = 32
latent_dim = 256

# Encoder
encoder_inputs = keras.layers.Input(shape=(None,))
enc_emb = keras.layers.Embedding(vocab_size_pt, embed_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = keras.layers.Input(shape=(None,))
dec_emb = keras.layers.Embedding(vocab_size_en, embed_dim, mask_zero=True)(decoder_inputs)
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_size_en, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Seq2Seq Model
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

# ---- Prepare Data for Training (teacher forcing) ----
def prepare_batch(src, tgt):
    decoder_inp = tgt[:, :-1]
    decoder_out = tgt[:, 1:]
    return (src, decoder_inp), decoder_out

train_dataset_tf = train_dataset.map(prepare_batch)
val_dataset_tf = val_dataset.map(prepare_batch)

# ---- Train ----
model.fit(train_dataset_tf, epochs=50, validation_data=val_dataset_tf)

# ---- Simple Inference Demo ----
def translate(sentence):
    start_token = tokenizer_en.vocab_size
    end_token = tokenizer_en.vocab_size + 1

    pt_tokens = [tokenizer_pt.vocab_size] + tokenizer_pt.encode(sentence.lower()) + [tokenizer_pt.vocab_size + 1]
    pt_tokens = keras.preprocessing.sequence.pad_sequences(
        [pt_tokens], maxlen=max_len, padding="post"
    )

    en_input = np.zeros((1, max_len), dtype=np.int64)
    en_input[0, 0] = start_token

    generated_tokens = []

    for t in range(max_len - 1):
        output_tokens = model.predict([pt_tokens, en_input], verbose=0)
        sampled_token = int(np.argmax(output_tokens[0, t, :]))

        if sampled_token == end_token:
            break

        if sampled_token < tokenizer_en.vocab_size:
            generated_tokens.append(sampled_token)

        en_input[0, t + 1] = sampled_token

    return tokenizer_en.decode(generated_tokens) if generated_tokens else ""

print("Portuguese: Olá")
print("English (predicted):", translate("Olá"))


c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...: 100%|██████████| 1/1 [00:00<00:00,  5.83 url/s]
Extraction completed...: 0 file [00:00, ? file/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]
Dl Completed...: 100%|██████████| 1/1 [00:00<00:00,  5.83 url/s]


TFDS download failed: Artifact http://www.phontron.com/data/qi18naacl-dataset.tar.gz, downloaded to C:\Users\Hait\tensorflow_datasets\downloads\ted_hrlr_translate\phontron.com_qi18naacl-datasetLVhDyvf9PQ-GjP4jim31ESZuvRjHI5wpNhR5SJvsNOo.tar.gz.tmp.3600ef179150498ba4be7b9581e9d062\index.html, has wrong checksum:
* Expected: UrlInfo(size=124.94 MiB, checksum='216a86c3df4d4f522856fe9b920ff5be6b394d769cc88974ae8f9f5546953bbc', filename='qi18naacl-dataset.tar.gz')
* Got: UrlInfo(size=705 bytes, checksum='dc6a6a54de630b2beee4b96c770b9f1b32615a8f0352ac31a1aab8285a351b8f', filename='index.html')
To debug, see: https://www.tensorflow.org/datasets/overview#fixing_nonmatchingchecksumerror
Using a small built-in Portuguese-English dataset instead.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 32)  │     10,240 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 32)  │     10,144 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    295,936 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    295,936 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 317) │     81,469 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

...

 Trainable params: 693,725 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
      1/Unknown 2s 2s/step - accuracy: 0.0000e+00 - loss: 5.7592

c:\Users\Hait\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 5.7592 - val_accuracy: 0.1200 - val_loss: 5.7522
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.1579 - loss: 5.7520 - val_accuracy: 0.2400 - val_loss: 5.7448
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.1805 - loss: 5.7452 - val_accuracy: 0.2400 - val_loss: 5.7361
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.1805 - loss: 5.7370 - val_accuracy: 0.2400 - val_loss: 5.7251
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.1805 - loss: 5.7263 - val_accuracy: 0.2400 - val_loss: 5.7104
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.1805 - loss: 5.7116 - val_accuracy: 0.2400 - val_loss: 5.6902
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.1805 - loss: 5.6905 - val_accuracy: 0.2400 - val_loss: 5.6620
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.1805 - loss: 5.6593 - val_accuracy: 0.2400 - val_loss: 5.6217
Epoch 9/5